# Llama3.1-8B base from Hugging Face

Run persiapan dengan **Internet ON**, CPU saja. Mengunduh base BF16 publik
`unsloth/Meta-Llama-3.1-8B`, repo yang tercantum pada konfigurasi adapter Llama Dhruv.
Revision dipin dan empat shard dicocokkan SHA256-nya dengan Hugging Face. Tidak ada training.
Keluaran `llama31-base/` dipakai sebagai input notebook inferensi Kaggle offline.
Lisensi base: Llama 3.1 Community License; kredit Meta dan Unsloth dipertahankan di README model.


In [ ]:
import os
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
os.environ['HF_HOME']='/tmp/ntl-llama-hf-cache'
os.environ.pop('HF_HUB_OFFLINE',None)
os.environ.pop('TRANSFORMERS_OFFLINE',None)
import hashlib,json,struct,time,threading
from pathlib import Path
from huggingface_hub import HfApi,snapshot_download
ROOT=Path('/kaggle/working/llama31-base')
ROOT.mkdir(exist_ok=True)
REPO='unsloth/Meta-Llama-3.1-8B'
REVISION='e9a141a2091ea561b96483212645a2a05e6f99fc'
start=time.time()
api=HfApi(token=False)
info=api.model_info(REPO,revision=REVISION,files_metadata=True,token=False)
assert info.sha==REVISION and not info.gated
selected=[item for item in info.siblings if item.rfilename.endswith(('.json','.safetensors')) or item.rfilename=='README.md']
print('Downloading public pretrained base',REPO,REVISION,len(selected),'files',flush=True)
weight_files={item.rfilename:getattr(getattr(item,'lfs',None),'size',0) for item in selected if item.rfilename.endswith('.safetensors')}
download_total=sum(weight_files.values())
stop=threading.Event()
def progress_monitor():
    while not stop.is_set():
        completed=[]
        for name,size in weight_files.items():
            path=ROOT/name
            if path.is_file() and path.stat().st_size==size:completed.append(name)
        confirmed=sum(weight_files[name] for name in completed)
        state={'confirmed_complete_shards':len(completed),'total_shards':len(weight_files),
               'confirmed_bytes':confirmed,'total_weight_bytes':download_total,
               'confirmed_percent_lower_bound':100*confirmed/download_total if download_total else 0}
        Path('/kaggle/working/asset-progress.json').write_text(json.dumps(state))
        print('DOWNLOAD_PROGRESS',json.dumps(state),flush=True)
        stop.wait(30)
thread=threading.Thread(target=progress_monitor,daemon=True)
thread.start()
try:
    snapshot_download(REPO,revision=REVISION,local_dir=str(ROOT),allow_patterns=[item.rfilename for item in selected],
                      token=False,max_workers=4)
finally:
    stop.set();thread.join(timeout=2)
print('DOWNLOAD FINISHED; verifying every weight shard against Hugging Face SHA256',flush=True)
cfg=json.loads((ROOT/'config.json').read_text())
assert (cfg['model_type'],cfg['hidden_size'],cfg['num_hidden_layers'],cfg['vocab_size'])==('llama',4096,32,128256)
assert not cfg.get('quantization_config'),'Expected original BF16 base, not a quantized variant'
index=json.loads((ROOT/'model.safetensors.index.json').read_text())
shards=set(index['weight_map'].values())
assert len(shards)==4
records={}
tensor_names=set()
for item in selected:
    path=ROOT/item.rfilename
    if not path.is_file():raise FileNotFoundError(item.rfilename)
    with path.open('rb') as stream:digest=hashlib.file_digest(stream,'sha256').hexdigest()
    lfs=getattr(item,'lfs',None)
    expected=getattr(lfs,'sha256',None) if lfs is not None else None
    if expected and digest!=expected:raise ValueError('HF checksum mismatch: '+item.rfilename)
    if item.rfilename in shards:
        assert expected,'Expected a published SHA256 for every weight shard'
        with path.open('rb') as stream:
            length=struct.unpack('<Q',stream.read(8))[0]
            header=json.loads(stream.read(length))
        tensor_names.update(name for name in header if name!='__metadata__')
    records[item.rfilename]={'bytes':path.stat().st_size,'sha256':digest,'hf_lfs_sha256':expected}
    print('Verified',item.rfilename,path.stat().st_size,flush=True)
assert set(index['weight_map'])==tensor_names
assert all((ROOT/name).is_file() for name in shards)
manifest={'repo':REPO,'revision':REVISION,'gated':False,'original_weight_dtype':'BF16','model_type':'llama',
          'hidden_size':4096,'num_hidden_layers':32,'files':records,'weight_shards':len(shards),
          'all_weight_shards_verified_against_hf':True,'seconds':time.time()-start,'training_performed':False}
(ROOT/'hf-model-manifest.json').write_text(json.dumps(manifest,indent=2))
Path('/kaggle/working/asset-report.json').write_text(json.dumps(manifest,indent=2))
print('ASSET READY',json.dumps({key:manifest[key] for key in ['repo','revision','weight_shards','seconds']}),flush=True)
